## 🤖 Streaming AI Tutor Chatbot with Gradio

Built a fully functional, real-time streaming chatbot using the OpenAI API and Gradio's `ChatInterface` — a practical, deployable AI tutor.

### What this demonstrates:
- **Streaming responses** — Token-by-token output via OpenAI's `stream=True`, yielding partial responses for a live "typing" effect instead of waiting for the full completion.
- **Backward-compatible history handling** — Supports both the newer dict-based message format (`{"role": ..., "content": ...}`) and legacy tuple-based history, making the function resilient across Gradio versions.
- **System prompt engineering** — Configured persona ("expert and helpful AI tutor") to steer response style and tone.
- **Environment-based config** — API keys managed securely via `.env` and `python-dotenv`, keeping secrets out of source code.
- **Error handling** — Wrapped the generation loop in try/except with logging, so failures degrade gracefully instead of crashing the UI.
- **Public shareable deployment** — Launched with `share=True` for instant demo links via Gradio's tunneling.

**Stack:** `OpenAI (gpt-4o-mini)` · `Gradio ChatInterface` · `python-dotenv`

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
if openai_api_key and openai_api_key.startswith("sk-proj") and len(openai_api_key) >10:
    print("OpenAI API key is valid")
else:
    print("OpenAI API key is invalid")

OpenAI API key is valid


In [5]:
openai_client = OpenAI(api_key=openai_api_key)

In [9]:
def ai_bot(message, history):
    try:
        history = [{"role": h['role'], "content":h['content']} for h in history]
        SYSTEM_PROMPT = "You are an expert and helpful AI tutor. Explain concepts clearly and concisely."
        message =[{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
        stream = openai_client.chat.completions.create(
            model="gpt-4o-mini-2024-07-18",
            max_tokens=2000,
            messages=message,
            temperature=0.5,
            stream=True)
        full_response = ""
        for chunks in stream:
            if chunks.choices[0].delta and chunks.choices[0].delta.content:
                partial_response =chunks.choices[0].delta.content
                full_response += partial_response
                yield full_response
    except Exception as e:
        print(e)
        print(f'There wan an error {e}')

In [10]:
"""
## Gradio ChatInterface Configuration

`gr.ChatInterface` wraps the `ai_bot` generator function and handles:
- Rendering the chat window
- Managing conversation history
- Displaying streaming output in real time
"""

gradio_ui = gr.ChatInterface(
    fn=ai_bot,
    title="AI Chatbot Chat",
    description="Ask anything to openAI gpt-4o mini LLM model",
    textbox=gr.Textbox(placeholder="Ask the AI....", container=False, scale=7),
    flagging_mode="never"
)
print("Launching AI ChatBot...")
gradio_ui.launch(share=True)

Launching AI ChatBot...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://10c5395b86e49a7635.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
